In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTENC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix
import seaborn as sns
from sklearn.impute import SimpleImputer
import numpy as np

In [2]:
glucose_file="/Users/tomliu/Downloads/glucose_2021-2023.xpt"
smoking_file="/Users/tomliu/Downloads/smoking-2021-2023.xpt"
diet1_file="/Users/tomliu/Downloads/diet1_2021-2023.xpt"
diet2_file="/Users/tomliu/Downloads/diet2_2021-2023.xpt"
demographics_file="/Users/tomliu/Downloads/demographic_2021-2023.xpt"
physical_file="/Users/tomliu/Downloads/physical_2021-2023.xpt"
weight_file="/Users/tomliu/Downloads/weight_2021-2023.xpt"
sleep_file="/Users/tomliu/Downloads/sleep_2021-2023.xpt"
alcohol_file="/Users/tomliu/Downloads/alcohol_2021-2023.xpt"

In [3]:
glucose_df = pd.read_sas(glucose_file, format="xport")[['SEQN','LBDGLUSI']]
smoking_df= pd.read_sas(smoking_file, format="xport")[['SEQN', 'SMQ020']]
diet1_df = pd.read_sas(diet1_file, format="xport")[['SEQN', 'DR1TKCAL', 'DR1TPROT', 'DR1TCARB','DR1TSUGR', 'DR1TFIBE','DR1TTFAT', 'DR1TCHOL']]
diet2_df = pd.read_sas(diet2_file, format="xport")[['SEQN', 'DR2TKCAL', 'DR2TPROT', 'DR2TCARB','DR2TSUGR', 'DR2TFIBE','DR2TTFAT', 'DR2TCHOL']]
demographics_df = pd.read_sas(demographics_file, format="xport")[['SEQN', 'RIAGENDR', 'RIDAGEYR','RIDRETH1', 'INDFMPIR']]
physical_df = pd.read_sas(physical_file, format="xport")[['SEQN','PAD680']]
weight_df = pd.read_sas(weight_file, format="xport")[['SEQN', 'WHD010', 'WHD020']]
sleep_df = pd.read_sas(sleep_file, format="xport")[['SEQN','SLD012']]
alcohol_df = pd.read_sas(alcohol_file, format="xport")[['SEQN', 'ALQ111']]

glucose_df.columns = ['ID', 'glucose']
smoking_df.columns=['ID', 'smoking']
diet1_df.columns=['ID', 'calories1', 'protein1','carbs1','sugar1','fiber1','fat1', 'cholesterol1']
diet2_df.columns=['ID', 'calories2', 'protein2','carbs2','sugar2','fiber2','fat2', 'cholesterol2']
demographics_df.columns = ['ID', 'gender', 'age', 'ethnicity', 'household income']
physical_df.columns=['ID', 'sedentary']
weight_df.columns=['ID', 'height', 'weight']
sleep_df.columns=['ID','hours of sleep']
alcohol_df.columns=['ID', 'ever had alcohol']

In [4]:
#drop missing glucose values
glucose_df=glucose_df.dropna()

#diagnose diabetes from 2 hour fasting period according to WHO guidelines
glucose_df['diagnosis'] = glucose_df['glucose'].apply(lambda x: 1 if x > 7.0 else 0)
glucose_df.drop(columns=['glucose'], inplace=True)

In [5]:
#merge all dataframes into combined dataframe
combined_df = glucose_df.merge(smoking_df, on='ID', how='left')\
                         .merge(diet1_df, on='ID', how='left')\
                         .merge(diet2_df, on='ID', how='left')\
                         .merge(demographics_df, on='ID', how='left')\
                         .merge(physical_df, on='ID', how='left')\
                         .merge(weight_df, on='ID', how='left')\
                         .merge(sleep_df, on='ID', how='left')\
                         .merge(alcohol_df, on='ID', how='left')

combined_df.drop(columns=['ID'], inplace=True)

In [6]:
#create a new column of average of two days of nutrient tracking
nutrient_pairs = [
    ('calories1', 'calories2'),
    ('protein1', 'protein2'),
    ('fiber1', 'fiber2'),
    ('carbs1', 'carbs2'),
    ('fat1', 'fat2'),
    ('sugar1', 'sugar2'),
    ('cholesterol1', 'cholesterol2')
]

# Create new averaged column
for new_col, (col1, col2) in zip(['avg_' + pair[0] for pair in nutrient_pairs], nutrient_pairs):
    if col1 in combined_df.columns and col2 in combined_df.columns:
        combined_df[new_col] = combined_df[[col1, col2]].mean(axis=1)
    else:
        print(f"Warning: Columns {col1} or {col2} not found - skipping {new_col}")

# Now delete the old columns
cols_to_drop = []
for col1, col2 in nutrient_pairs:
    if col1 in combined_df.columns:
        cols_to_drop.append(col1)
    if col2 in combined_df.columns:
        cols_to_drop.append(col2)

combined_df = combined_df.drop(columns=cols_to_drop)

# Verify the new columns
print(combined_df.to_string())

      diagnosis  smoking  gender   age  ethnicity  household income     sedentary  height  weight  hours of sleep  ever had alcohol  avg_calories1  avg_protein1    avg_fiber1    avg_carbs1      avg_fat1    avg_sugar1  avg_cholesterol1
0             0      1.0     1.0  43.0        5.0      5.000000e+00  3.600000e+02    71.0   190.0             9.5               NaN   1.390000e+03  8.852500e+01  5.950000e+00  1.097850e+02  4.742000e+01  2.206000e+01      2.910000e+02
1             0      1.0     1.0  66.0        3.0      5.000000e+00  4.800000e+02    70.0   220.0             9.0               1.0   2.484500e+03  8.262000e+01  2.415000e+01  2.804400e+02  7.319000e+01  1.134950e+02      1.610000e+02
2             1      2.0     2.0  44.0        2.0      1.410000e+00  2.400000e+02    60.0   150.0             8.0               1.0   1.735000e+03  6.711500e+01  2.140000e+01  2.453250e+02  5.568500e+01  1.109200e+02      4.040000e+02
3             0      1.0     1.0  34.0        1.0      1.330

In [7]:
cat_columns = ['gender', 'age','ethnicity', 'ever had alcohol', 'smoking']
num_columns = [col for col in combined_df.columns if col not in cat_columns + ['diagnosis']]

#impute the data using average
# For numerical features
num_imputer = SimpleImputer(strategy='mean')
combined_df[num_columns] = num_imputer.fit_transform(combined_df[num_columns])

# For categorical features
cat_imputer = SimpleImputer(strategy='most_frequent')
combined_df[cat_columns] = cat_imputer.fit_transform(combined_df[cat_columns])


In [8]:
# Data and target
X = combined_df[cat_columns + num_columns].values
y = combined_df['diagnosis'].values

# Initial train-test split (no leakage)
X_train, X_test, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [9]:
# Convert data to PyTorch tensors
X_cat_train = torch.tensor(X_train[:, :len(cat_columns)], dtype=torch.long)
X_num_train = torch.tensor(X_train[:, len(cat_columns):], dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)

X_cat_test = torch.tensor(X_test[:, :len(cat_columns)], dtype=torch.long)
X_num_test = torch.tensor(X_test[:, len(cat_columns):], dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32).unsqueeze(1)

category_sizes = [len(np.unique(X_train[:, i])) for i in range(len(cat_columns))]

In [10]:

# Define TabTransformer model
class TabTransformer(nn.Module):
    def __init__(self, categories, num_continuous, dim=32, depth=2, heads=4, dropout=0.1):
        super().__init__()
        self.cat_embeds = nn.ModuleList([nn.Embedding(cat_size, dim) for cat_size in categories])
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=dim, nhead=heads, dropout=dropout, batch_first=True),
            num_layers=depth
        )
        self.cont_bn = nn.BatchNorm1d(num_continuous) if num_continuous > 0 else None
        self.mlp = nn.Sequential(
            nn.Linear(dim * len(categories) + num_continuous, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 1)
        )

    def forward(self, x_cat, x_cont):
        cat_embeds = [emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeds)]
        x_cat = torch.stack(cat_embeds, dim=1)
        x_cat = self.transformer(x_cat).flatten(1)

        if x_cont is not None and self.cont_bn:
            x_cont = self.cont_bn(x_cont)

        x = torch.cat([x_cat, x_cont], dim=1) if x_cont is not None else x_cat
        return self.mlp(x)

# Instantiate model
model = TabTransformer(categories=category_sizes, num_continuous=len(num_columns))

In [11]:
# Loss function (correctly instantiate it)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_losses = []

# Training loop
for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_cat_train, X_num_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    print(f"Epoch {epoch+1:02} | Loss: {loss.item():.4f}")


IndexError: index out of range in self

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses)+1), train_losses, marker='o')
plt.title("Training Loss vs. Epoch")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross Entropy Loss")
plt.grid(True)
plt.show()

In [ ]:
import torch.nn.functional as F

# Evaluation mode
model.eval()
with torch.no_grad():
    logits = model(X_cat_test, X_num_test)
    probs = torch.sigmoid(logits).squeeze()
    preds = (probs > 0.57).int()

# Convert to NumPy
y_true = y_test.squeeze().numpy()
y_pred = preds.numpy()
y_prob = probs.numpy()

# Accuracy scores
acc = accuracy_score(y_true, y_pred)
precision =  precision_score(y_true, y_pred)
recall =  recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")


# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["No Diabetes", "Diabetes"], yticklabels=["No Diabetes", "Diabetes"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(loc="lower right")
plt.show()